In [6]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

BASE = '/content/drive/MyDrive/data-analytics-portfolio/hotel-cancellation-analysis'
df = pd.read_csv(f'{BASE}/data/raw/hotel_bookings.csv')

print(df.shape)
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(119390, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [7]:
df.info()
print(df.isna().sum()[df.isna().sum() > 0])
print('Duplicates:', df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal            

In [8]:
cleaning_log = []
initial_rows = len(df)
cleaning_log.append(('Initial rows', 0, initial_rows))

# Working copy: the raw dataframe stays untouched
clean = df.copy()

# 1. Missing values: children -> 0, country -> 'Unknown'
clean['children'] = clean['children'].fillna(0).astype(int)
clean['country'] = clean['country'].fillna('Unknown')

# 2. Bookings with no guests
mask = (clean['adults'] + clean['children'] + clean['babies']) == 0
cleaning_log.append(('No guests', mask.sum(), len(clean) - mask.sum()))
clean = clean[~mask]

# 3. ADR <= 0
mask = clean['adr'] <= 0
cleaning_log.append(('ADR <= 0', mask.sum(), len(clean) - mask.sum()))
clean = clean[~mask]

# 4. Zero nights
mask = (clean['stays_in_week_nights'] + clean['stays_in_weekend_nights']) == 0
cleaning_log.append(('Zero nights', mask.sum(), len(clean) - mask.sum()))
clean = clean[~mask]

# New features
clean['total_nights'] = clean['stays_in_week_nights'] + clean['stays_in_weekend_nights']
clean['revenue'] = clean['adr'] * clean['total_nights']

cleaning_log_df = pd.DataFrame(
    cleaning_log, columns=['rule', 'rows_removed', 'rows_remaining']
)
cleaning_log_df

,rule,rows_removed,rows_remaining
0,Initial rows,0,119390
1,No guests,180,119210
2,ADR <= 0,1811,117399
3,Zero nights,0,117399


In [10]:
clean.to_csv(f'{BASE}/data/clean/clean_bookings.csv', index=False)
print(clean.shape)

(117399, 34)


In [11]:
# Arrival date as a real date
clean['arrival_date'] = pd.to_datetime(
    clean['arrival_date_year'].astype(str) + '-' +
    clean['arrival_date_month'] + '-' +
    clean['arrival_date_day_of_month'].astype(str),
    format='%Y-%B-%d'
)

# Lead time groups
clean['lead_time_group'] = pd.cut(
    clean['lead_time'],
    bins=[-1, 7, 30, 90, 180, float('inf')],
    labels=['0-7 days', '8-30 days', '31-90 days', '91-180 days', '180+ days']
)

# Readable booking status
clean['booking_status'] = clean['is_canceled'].map({0: 'Not canceled', 1: 'Canceled'})

# Save the final clean file (overwrites the previous one)
clean.to_csv(f'{BASE}/data/clean/clean_bookings.csv', index=False)
print(clean.shape)

(117399, 37)
